[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kauelopes/topicos-ml/blob/main/aulas/02-python-numpy-jax/notebooks/02_demo.ipynb)

# Aula 2 — Computação aplicada: de Python ao aprendizado de máquina
*Computação Aplicada à Economia: Tópicos em Aprendizado de Máquina*

**A pergunta da aula:** um programa de treinamento para o emprego aumentou a renda de quem participou?

Vamos responder a essa pergunta com dados reais (o experimento **NSW**, estudado por LaLonde, 1986) e, no caminho, construir a caixa de ferramentas do curso:

| Parte | Ferramenta | O que vamos fazer |
|---|---|---|
| 1 | Python | variáveis, listas, dicionários, `if`, `for`, funções |
| 2 | NumPy | vetores, máscaras, matrizes e o MQO em uma linha |
| 3 | pandas | ler a base, filtrar, agrupar, comparar e plotar |
| 4 | avaliação | diferença de médias, MQO com *dummy* e o grupo de comparação errado |
| 5 | JAX | derivadas automáticas e **gradiente descendente** para minimizar o EQM |
| 6 | scikit-learn | regressão, floresta aleatória e "dá para prever quem foi tratado?" |

**Como usar:** `Shift+Enter` executa a célula e avança. Nas células **✍️ Sua vez**, troque o `...` por código; a resposta fica escondida logo abaixo.

## 0. Configuração (uma vez por notebook)
Importamos as bibliotecas e definimos o visual dos gráficos **uma vez só**. `CORES` e `plt.rcParams` são *dicionários*: pares nome → valor (voltaremos a eles na Parte 1).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CORES = {"azul": "#2A6F97", "laranja": "#EB811B", "verde": "#14B03D",
         "vermelho": "#B22234", "escuro": "#23373B", "cinza": "#8C8C8C"}

plt.rcParams.update({
    "figure.figsize": (7, 4),
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "legend.frameon": False,
})
pd.set_option("display.precision", 2)

## 1. Python essencial: um participante, depois cinco

### 1.1 Variáveis e tipos
Cada variável guarda **um valor** com um **tipo**. Abaixo, o primeiro participante da base que vamos usar.

In [ ]:
idade = 37              # int: número inteiro
anos_estudo = 11        # int
renda_1978 = 9930.05    # float: número com casas decimais (US$ por ano)
participou = True       # bool: verdadeiro ou falso
programa = "NSW"        # str: texto

print(type(idade), type(renda_1978), type(participou), type(programa))

Operações e **f-strings** (o `f` antes das aspas permite colocar variáveis dentro do texto, entre `{}`; depois dos dois-pontos vem o formato):

In [ ]:
renda_mensal = renda_1978 / 12
print(f"Renda mensal em 1978: US$ {renda_mensal:,.2f}")
print(f"Daqui a 10 anos terá {idade + 10} anos")
print(f"Tem ensino médio completo? {anos_estudo >= 12}")

### 1.2 Listas: vários valores em ordem
A contagem começa em **zero**. Índices negativos contam do fim.

In [ ]:
rendas = [9930.05, 3595.89, 24909.45, 2113.72, 7618.64]

print(rendas[0], rendas[-1])   # primeiro e último
print(rendas[1:3])             # fatia: posições 1 e 2 (o 3 fica de fora)
print(len(rendas), sum(rendas), max(rendas))

rendas.append(0.0)             # acrescenta no fim
rendas

### 1.3 Dicionários: valores com nome
Uma **ficha**: cada informação é acessada pelo nome (a *chave*), e não pela posição.

In [ ]:
pessoa = {"idade": 37, "educ": 11, "tratado": True, "re78": 9930.05}

print(pessoa["re78"])
pessoa["casado"] = False       # nova chave
pessoa

Uma **lista de dicionários** já é uma pequena tabela: cada dicionário é uma linha. Estes são cinco participantes reais da base (três sorteados para o programa e dois para o grupo de controle).

In [ ]:
pessoas = [
    {"idade": 37, "educ": 11, "tratado": True,  "re78": 9930.05},
    {"idade": 22, "educ": 9,  "tratado": True,  "re78": 3595.89},
    {"idade": 30, "educ": 12, "tratado": True,  "re78": 24909.45},
    {"idade": 34, "educ": 12, "tratado": False, "re78": 2113.72},
    {"idade": 24, "educ": 10, "tratado": False, "re78": 7618.64},
]
pessoas[2]["re78"]

### 1.4 Decisões (`if`) e repetições (`for`)
Atenção à **indentação** (os 4 espaços): é ela que diz o que está dentro do `if` e do `for`.

In [ ]:
for p in pessoas:
    if p["re78"] == 0:
        faixa = "sem renda"
    elif p["re78"] < 5000:
        faixa = "baixa"
    else:
        faixa = "média ou alta"
    grupo = "tratado" if p["tratado"] else "controle"
    print(f"{grupo:>8} | {p['idade']} anos | US$ {p['re78']:>9,.2f} | {faixa}")

Agora a primeira **avaliação** do curso, ainda na mão: renda média dos tratados menos a dos controles.

In [ ]:
soma_t, n_t = 0.0, 0
soma_c, n_c = 0.0, 0
for p in pessoas:
    if p["tratado"]:
        soma_t += p["re78"]     # o mesmo que soma_t = soma_t + p["re78"]
        n_t += 1
    else:
        soma_c += p["re78"]
        n_c += 1

print(f"Tratados: {soma_t / n_t:,.0f} | Controles: {soma_c / n_c:,.0f}")
print(f"Diferença: {soma_t / n_t - soma_c / n_c:,.0f}")

### 1.5 Funções e *list comprehensions*
Se vamos repetir um cálculo, damos um **nome** a ele com `def`. A *list comprehension* monta uma lista numa linha: `[o_que for item in lista if condição]`.

In [ ]:
def media(valores):
    """Média aritmética de uma lista de números."""
    return sum(valores) / len(valores)


def diferenca_medias(pessoas):
    tratados = [p["re78"] for p in pessoas if p["tratado"]]
    controles = [p["re78"] for p in pessoas if not p["tratado"]]
    return media(tratados) - media(controles)


diferenca_medias(pessoas)

**Com 5 pessoas, essa diferença não quer dizer nada**: basta trocar uma pessoa para o número mudar muito. Precisamos da base inteira, e para isso precisamos de ferramentas melhores.

#### ✍️ Sua vez
Calcule a **idade média dos controles** usando `media` e uma *list comprehension*.

In [ ]:
idade_media_controles = ...
idade_media_controles

<details><summary><b>Ver uma resposta</b> (tente antes!)</summary>

```python
idade_media_controles = media([p["idade"] for p in pessoas if not p["tratado"]])
```
</details>

## 2. NumPy: contas com vetores inteiros

O laço da Parte 1 funciona, mas é longo e lento. O NumPy faz a mesma conta **de uma vez** sobre o vetor todo (*vetorização*).

In [ ]:
re78 = np.array([p["re78"] for p in pessoas])
tratado = np.array([p["tratado"] for p in pessoas])

print(re78)
print(re78 / 1000)          # divide TODOS os elementos, sem laço
print(re78.mean(), re78.shape, re78.dtype)

### 2.1 Máscaras booleanas
Um vetor de `True`/`False` seleciona elementos. O `~` inverte a máscara. Compare com o laço de 10 linhas da Parte 1:

In [ ]:
print(re78[tratado])
re78[tratado].mean() - re78[~tratado].mean()

### 2.2 Por que vetorizar? Velocidade
Vamos simular 1 milhão de rendas e comparar o laço com o NumPy. `np.random.default_rng(seed)` cria um gerador de números aleatórios **reprodutível**.

In [ ]:
import time

rng = np.random.default_rng(42)
rendas_sim = rng.exponential(scale=5000, size=1_000_000)

inicio = time.perf_counter()
total = 0.0
for r in rendas_sim:
    total += r
media_laco = total / len(rendas_sim)
t_laco = time.perf_counter() - inicio

inicio = time.perf_counter()
media_np = rendas_sim.mean()
t_np = time.perf_counter() - inicio

print(f"Laço:  {t_laco:.3f} s | NumPy: {t_np:.4f} s | {t_laco / t_np:,.0f}x mais rápido")

### 2.3 Matrizes e o MQO em uma linha
Uma matriz é um array com duas dimensões (`shape` = linhas × colunas). O `@` multiplica matrizes e `.T` transpõe. O estimador de MQO

$$\hat\beta = (X^\top X)^{-1} X^\top y$$

vira uma linha: `np.linalg.solve(X.T @ X, X.T @ y)` (resolver o sistema é mais estável do que inverter a matriz).

In [ ]:
X = np.column_stack([np.ones(5), tratado])   # coluna de 1s (intercepto) e a dummy
print(X, X.shape, sep="\n")

beta = np.linalg.solve(X.T @ X, X.T @ re78)
beta

Repare: o **intercepto** é a média dos controles e o **coeficiente da dummy** é exatamente a diferença de médias que calculamos com o laço. Guardem esse fato: vamos usá-lo com a base inteira.

#### ✍️ Sua vez
Que **fração** dos 5 participantes teve renda acima de US$ 5.000? Dica: a média de um vetor booleano é a proporção de `True`.

In [ ]:
fracao = ...
fracao

<details><summary><b>Ver uma resposta</b> (tente antes!)</summary>

```python
fracao = (re78 > 5000).mean()
```
</details>

## 3. pandas: a base do LaLonde

### O programa e os dados
- O **National Supported Work Demonstration (NSW)** foi um programa americano dos anos 1970 que oferecia emprego subsidiado e acompanhamento a pessoas com muita dificuldade de inserção no mercado de trabalho.
- A vaga foi **sorteada** entre os candidatos: quem foi sorteado é **tratado**, quem não foi é **controle**. É um experimento aleatorizado.
- **LaLonde (1986)** usou o experimento para testar os métodos econométricos da época: se esquecermos o sorteio e compararmos os tratados com pessoas de uma pesquisa domiciliar (CPS ou PSID), os métodos recuperam o efeito verdadeiro?
- Usamos a subamostra de homens de **Dehejia e Wahba (1999)**, que mantém quem tem a renda de 1974 informada.

| Variável | Significado |
|---|---|
| `treat` | 1 se foi sorteado para o programa |
| `age`, `education` | idade e anos de estudo |
| `black`, `hispanic`, `married` | dummies de raça/etnia e estado civil |
| `nodegree` | 1 se não concluiu o ensino médio |
| `re74`, `re75`, `re78` | renda anual em 1974, 1975 (antes) e 1978 (depois do programa), em US$ |

### Leitura (com plano B)
Os arquivos ficam na página de Rajeev Dehejia no NBER. `pd.read_stata` lê arquivos do Stata (`.dta`) direto da internet. Se a internet falhar, o `except` gera uma base **simulada** com o mesmo formato, para a aula não travar.

In [ ]:
URL = "https://users.nber.org/~rdehejia/data/"
COVARIAVEIS = ["age", "education", "black", "hispanic", "married", "nodegree", "re74", "re75"]


def gerar_base_simulada(n, tratado, seed):
    """Plano B: participantes com perfil parecido com o do NSW (tratado=0/1)
    ou com o da CPS (tratado=None, controles da pesquisa domiciliar)."""
    rng = np.random.default_rng(seed)
    cps = tratado is None
    df = pd.DataFrame({
        "age": rng.integers(17, 55, n),
        "education": rng.integers(8 if cps else 4, 17, n),
        "black": rng.binomial(1, 0.07 if cps else 0.84, n),
        "hispanic": rng.binomial(1, 0.07, n),
        "married": rng.binomial(1, 0.71 if cps else 0.17, n),
    })
    df["nodegree"] = (df["education"] < 12).astype(int)
    escala = 14_000 if cps else 2_000
    df["re74"] = rng.exponential(escala, n) * rng.binomial(1, 0.88 if cps else 0.3, n)
    df["re75"] = 0.8 * df["re74"] + rng.exponential(escala * 0.2, n)
    df["treat"] = 0 if cps else tratado
    efeito = 0 if cps else 1800 * tratado
    df["re78"] = np.maximum(0, 0.6 * df["re75"] + 300 * df["education"]
                            + efeito + rng.normal(0, 5000, n))
    return df


try:
    nsw = pd.read_stata(URL + "nsw_dw.dta")
    cps = pd.read_stata(URL + "cps_controls.dta")
    print("Dados reais carregados do NBER.")
except Exception as erro:
    print(f"Sem acesso ({erro}). Usando dados SIMULADOS: os números vão diferir do texto.")
    nsw = pd.concat([gerar_base_simulada(185, 1, 1), gerar_base_simulada(260, 0, 2)])
    cps = gerar_base_simulada(15_992, None, 3)

nsw.head()

### 3.1 Primeiro contato com um `DataFrame`
Um **DataFrame** é uma tabela: colunas com nome e tipo (cada coluna é um vetor NumPy por dentro).

In [ ]:
print(nsw.shape)          # (linhas, colunas)
nsw.dtypes

In [ ]:
nsw.describe().T          # resumo de cada coluna; .T deita a tabela

### 3.2 Selecionar colunas e filtrar linhas
- `df["col"]` devolve uma coluna; `df[["a", "b"]]`, várias.
- `df[condição]` filtra linhas com uma **máscara booleana**, como no NumPy.

In [ ]:
nsw[["age", "education", "re78"]].head(3)

In [ ]:
jovens_sem_diploma = nsw[(nsw["age"] < 25) & (nsw["nodegree"] == 1)]
print(len(jovens_sem_diploma), "participantes com menos de 25 anos e sem ensino médio")

### 3.3 Criar colunas
Toda a coluna é calculada de uma vez (vetorização de novo).

In [ ]:
nsw["re78_mil"] = nsw["re78"] / 1000          # renda em milhares de US$
nsw["sem_renda_78"] = nsw["re78"] == 0         # bool
nsw["grupo"] = nsw["treat"].map({1: "Tratados", 0: "Controles"})

print(f"{nsw['sem_renda_78'].mean():.1%} não tiveram renda em 1978")
nsw[["treat", "grupo", "re78", "re78_mil", "sem_renda_78"]].head(3)

### 3.4 Agrupar e resumir: `groupby`
"Separe por grupo, calcule, junte" numa linha só: a operação mais usada na análise de dados.

In [ ]:
nsw.groupby("grupo")["re78"].agg(["count", "mean", "median"])

**Tabela de balanceamento.** Num experimento bem feito, os grupos se parecem *antes* do programa: o sorteio cuida disso.

In [ ]:
balanco = nsw.groupby("grupo")[COVARIAVEIS].mean().T
balanco["diferença"] = balanco["Tratados"] - balanco["Controles"]
balanco

**Pergunta:** olhe `re74` e `re75` (rendas antes do programa). Os grupos eram parecidos? E depois, em 1978?

### 3.5 Gráficos
Dois gráficos que contam a história: a distribuição da renda em 1978 e a média por grupo.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 3.8))

cores_grupo = {"Controles": CORES["cinza"], "Tratados": CORES["laranja"]}
bins = np.arange(0, 41, 2.5)
for nome, dados in nsw.groupby("grupo"):
    axs[0].hist(dados["re78_mil"], bins=bins, alpha=0.6, density=True,
                color=cores_grupo[nome], label=nome)
axs[0].set(title="Renda em 1978", xlabel="mil US$", ylabel="densidade")
axs[0].legend()

medias = nsw.groupby("grupo")["re78_mil"].mean()
barras = axs[1].bar(medias.index, medias.values,
                    color=[cores_grupo[g] for g in medias.index])
axs[1].bar_label(barras, fmt="%.2f mil")
axs[1].set(title="Renda média em 1978", ylabel="mil US$")
axs[1].grid(axis="x", visible=False)

fig.tight_layout()

#### ✍️ Sua vez
Calcule a renda média em 1978 por grupo **e** por `nodegree` (sem ensino médio). Dica: `groupby` aceita uma lista de colunas.

In [ ]:
...

<details><summary><b>Ver uma resposta</b> (tente antes!)</summary>

```python
nsw.groupby(["grupo", "nodegree"])["re78"].mean().unstack()
```
</details>

## 4. Avaliação: o programa funcionou?

### 4.1 Diferença de médias = MQO com uma dummy
Como o tratamento foi **sorteado**, a diferença de médias estima o efeito do programa.

In [ ]:
medias = nsw.groupby("treat")["re78"].mean()
efeito = medias[1] - medias[0]
print(f"Efeito estimado do programa: US$ {efeito:,.0f} por ano")

Com os 5 participantes da Parte 1, a diferença dava cerca de US\$ 7.946; com os 445, dá US\$ 1.794. Amostras pequenas enganam.

E o mesmo número sai do MQO de `re78` numa constante e na dummy `treat`, com a linha de NumPy da Parte 2:

In [ ]:
y = nsw["re78"].to_numpy()
X = np.column_stack([np.ones(len(nsw)), nsw["treat"]])
beta = np.linalg.solve(X.T @ X, X.T @ y)
print(f"intercepto = {beta[0]:,.0f} (média dos controles) | coef. de treat = {beta[1]:,.0f}")

#### Por que o coeficiente da dummy é a diferença de médias?

O modelo é $y_i = a + b\,D_i + \varepsilon_i$, com $D_i = 1$ para tratados e $D_i = 0$ para controles. Ele só faz **duas previsões**: $\hat y = a$ para os controles e $\hat y = a + b$ para os tratados.

O MQO minimiza a soma dos quadrados dos resíduos:

$$
\min_{a,\,b}\; S(a,b) = \sum_{i=1}^{n} \big(y_i - a - b\,D_i\big)^2 .
$$

As condições de primeira ordem são

$$
\begin{aligned}
\text{(I)}\quad \frac{\partial S}{\partial a} = 0 &\;\Longrightarrow\; \sum_{i=1}^{n} \big(y_i - a - b\,D_i\big) = 0, \\
\text{(II)}\quad \frac{\partial S}{\partial b} = 0 &\;\Longrightarrow\; \sum_{i=1}^{n} D_i\,\big(y_i - a - b\,D_i\big) = 0 .
\end{aligned}
$$

**Passo 1.** Em (II), o fator $D_i$ zera os termos dos controles. Sobram só os $n_1$ tratados:

$$
\sum_{i:\,D_i=1} \big(y_i - a - b\big) = 0
\;\Longrightarrow\; n_1\,\bar y_1 - n_1\,(a + b) = 0
\;\Longrightarrow\; a + b = \bar y_1 .
$$

**Passo 2.** Separe a soma de (I) em controles e tratados. A parte dos tratados é exatamente a soma do Passo 1, que vale zero:

$$
\sum_{i:\,D_i=0} \big(y_i - a\big) \;+\; \underbrace{\sum_{i:\,D_i=1} \big(y_i - a - b\big)}_{=\,0} = 0
\;\Longrightarrow\; n_0\,\bar y_0 - n_0\,a = 0
\;\Longrightarrow\; a = \bar y_0 .
$$

**Passo 3.** Subtraindo:

$$
\boxed{\;\hat a = \bar y_0, \qquad \hat b = \bar y_1 - \bar y_0\;}
$$

**Intuição:** a constante que minimiza a soma dos quadrados dos desvios de um grupo é a **média** desse grupo. Com intercepto e dummy, o modelo tem um parâmetro livre por grupo, então cada previsão vira a média do grupo e o coeficiente da dummy é a distância entre as duas médias.

**O mesmo resultado pela covariância.** Com $p = n_1/n$, temos $\operatorname{Var}(D) = p(1-p)$ e $\operatorname{Cov}(y, D) = p(1-p)\,(\bar y_1 - \bar y_0)$. Logo

$$
\hat b = \frac{\operatorname{Cov}(y, D)}{\operatorname{Var}(D)} = \bar y_1 - \bar y_0 .
$$

> **Ligação com a Parte 5:** as derivadas (I) e (II) são, a menos do fator $-2/n$, o gradiente do EQM que o JAX vai calcular. O gradiente descendente para quando esse gradiente zera, ou seja, exatamente em $a = \bar y_0$ e $b = \bar y_1 - \bar y_0$.

Conferindo cada peça com os dados:

In [ ]:
D = nsw["treat"].to_numpy()
y0_barra = y[D == 0].mean()
y1_barra = y[D == 1].mean()
p = D.mean()

print(f"média dos controles  = {y0_barra:,.2f} | intercepto a      = {beta[0]:,.2f}")
print(f"média dos tratados   = {y1_barra:,.2f} | a + b             = {beta[0] + beta[1]:,.2f}")
print(f"diferença de médias  = {y1_barra - y0_barra:,.2f} | coeficiente b     = {beta[1]:,.2f}")

cov_yD = np.mean((y - y.mean()) * (D - p))
print(f"Cov(y, D) / Var(D)   = {cov_yD / (p * (1 - p)):,.2f}")

Acrescentar as covariáveis quase não muda o resultado: com sorteio, elas não estão correlacionadas com o tratamento.

In [ ]:
X_cov = np.column_stack([np.ones(len(nsw)), nsw["treat"], nsw[COVARIAVEIS]])
beta_cov = np.linalg.lstsq(X_cov, y, rcond=None)[0]
print(f"Coeficiente de treat com covariáveis: US$ {beta_cov[1]:,.0f}")

### 4.2 E se não houvesse sorteio?
A pergunta de LaLonde: e se tivéssemos só os tratados e precisássemos de um grupo de comparação "da prateleira"? Vamos trocar os controles do experimento por homens da **CPS**, uma grande pesquisa domiciliar americana. `pd.concat` empilha tabelas.

In [ ]:
nao_exp = pd.concat([nsw[nsw["treat"] == 1], cps], ignore_index=True)
nao_exp["grupo"] = nao_exp["treat"].map({1: "Tratados (NSW)", 0: "CPS"})

medias_ne = nao_exp.groupby("treat")["re78"].mean()
print(f"{len(cps):,} controles da CPS")
print(f"'Efeito' com a CPS: US$ {medias_ne[1] - medias_ne[0]:,.0f}")

**O sinal inverteu.** O programa parece *destruir* renda. Por quê? Olhe quem são as pessoas:

In [ ]:
nao_exp.groupby("grupo")[COVARIAVEIS].mean().T

In [ ]:
fig, ax = plt.subplots()
bins = np.arange(0, 41, 2)
ax.hist(cps["re75"] / 1000, bins=bins, density=True, alpha=0.6,
        color=CORES["azul"], label="CPS")
ax.hist(nsw.loc[nsw["treat"] == 1, "re75"] / 1000, bins=bins, density=True,
        alpha=0.6, color=CORES["laranja"], label="Tratados (NSW)")
ax.set(title="Renda em 1975, antes do programa", xlabel="mil US$", ylabel="densidade")
ax.legend();

Os homens da CPS são mais velhos, mais escolarizados, mais casados e ganhavam **muito mais** antes do programa. Comparar os dois grupos mede essas diferenças, e não o efeito do programa: é **viés de seleção**.

Será que controlar pelas covariáveis numa regressão resolve?

In [ ]:
y_ne = nao_exp["re78"].to_numpy()
X_ne = np.column_stack([np.ones(len(nao_exp)), nao_exp["treat"], nao_exp[COVARIAVEIS]])
beta_ne = np.linalg.lstsq(X_ne, y_ne, rcond=None)[0]
print(f"Coeficiente de treat com covariáveis (CPS): US$ {beta_ne[1]:,.0f}")

| Comparação | Efeito estimado |
|---|---|
| Experimento (diferença de médias) | **≈ US\$ 1.794** |
| Tratados × CPS, sem controles | ≈ −US\$ 8.498 |
| Tratados × CPS, MQO com covariáveis | ≈ US\$ 699 |

Controlar ajuda muito, mas ainda erra bastante. Esse é o resultado que tornou o trabalho de LaLonde famoso. **Na Aula 6** voltaremos aqui com ferramentas de aprendizado de máquina para tentar chegar mais perto do experimento.

## 5. JAX: a derivada que treina modelos

Na Parte 4 o MQO saiu de uma fórmula fechada. A maioria dos modelos de ML **não tem fórmula**: eles são treinados **descendo o morro** da função de perda, passo a passo. Para isso precisamos de derivadas, e o JAX calcula derivadas de qualquer função escrita em Python.

### 5.1 Uma derivada simples
Para $f(b) = (b - 3)^2$, temos $f'(b) = 2(b - 3)$, então $f'(0) = -6$.

In [ ]:
import jax
import jax.numpy as jnp       # mesma interface do NumPy


def f(b):
    return (b - 3) ** 2


df = jax.grad(f)               # grad recebe uma função e devolve a função derivada
print(df(0.0), df(3.0), df(5.0))

### 5.2 Gradiente descendente na mão
A derivada aponta para onde a função **sobe**. Andando um pouco no sentido contrário, ela desce:

$$b_{\text{novo}} = b - \eta \, f'(b)$$

em que $\eta$ (eta) é a **taxa de aprendizado**, o tamanho do passo.

In [ ]:
b = 0.0
eta = 0.1
for passo in range(8):
    b = b - eta * df(b)
    print(f"passo {passo + 1}: b = {b:.4f}, f(b) = {f(b):.4f}")

### 5.3 Minimizando o EQM do nosso modelo
Agora o problema de verdade: o modelo $\widehat{re78} = a + b \cdot treat$ (renda em mil US\$) e a perda é o **erro quadrático médio**

$$\text{EQM}(a, b) = \frac{1}{n}\sum_{i=1}^{n} \big(y_i - a - b \, treat_i\big)^2 .$$

O MQO minimiza exatamente essa função. Vamos chegar ao mesmo lugar sem fórmula.

In [ ]:
y_j = jnp.array(nsw["re78_mil"].to_numpy())
t_j = jnp.array(nsw["treat"].to_numpy())


def eqm(params):
    a, b = params
    previsao = a + b * t_j
    return jnp.mean((y_j - previsao) ** 2)


grad_eqm = jax.grad(eqm)
params = jnp.array([0.0, 0.0])
print("EQM em (0, 0):", eqm(params))
print("gradiente do JAX:", grad_eqm(params))

Conferindo com a derivada feita à mão: $\partial\,\text{EQM}/\partial a = -\tfrac{2}{n}\sum (y_i - a - b\,t_i)$ e $\partial\,\text{EQM}/\partial b = -\tfrac{2}{n}\sum t_i (y_i - a - b\,t_i)$.

In [ ]:
a, b = params
residuo = y_j - a - b * t_j
print("gradiente à mão: ", jnp.array([-2 * residuo.mean(), -2 * (t_j * residuo).mean()]))

O laço de treino: o mesmo da Parte 5.2, agora com dois parâmetros. Guardamos o EQM a cada passo para ver o modelo aprender.

In [ ]:
def treinar(eta, passos=60):
    params = jnp.array([0.0, 0.0])
    historico = []
    for _ in range(passos):
        params = params - eta * grad_eqm(params)
        historico.append(float(eqm(params)))
    return params, historico


params, hist = treinar(eta=0.5)
print(f"a = {params[0]:.3f} mil, b = {params[1]:.3f} mil")
print(f"MQO (Parte 4): a = {beta[0] / 1000:.3f} mil, b = {beta[1] / 1000:.3f} mil")

O gradiente descendente chegou ao **mesmo efeito do programa** que o MQO. E se mudarmos o tamanho do passo?

In [ ]:
fig, ax = plt.subplots()
for eta, cor in [(0.1, CORES["azul"]), (0.5, CORES["verde"]), (1.0, CORES["vermelho"])]:
    _, hist = treinar(eta, passos=30)
    ax.plot(range(1, 31), hist, marker="o", ms=3, color=cor, label=f"η = {eta}")
ax.set_yscale("log")
ax.set_ylim(40, 1e4)
ax.set(title="Treino por gradiente descendente", xlabel="passo", ylabel="EQM (escala log)")
ax.legend();

- **η = 0,1**: desce, mas devagar.
- **η = 0,5**: converge em poucos passos.
- **η = 1,0**: o passo é grande demais, o modelo "pula" o vale e **diverge** (o EQM explode).

Escolher o passo é uma das artes do treino de modelos. Tema da **Aula 4**.

#### ✍️ Sua vez
Rode `treinar` com `eta=0.05` e 60 passos. A que valor de `b` ele chega? Por que é diferente?

In [ ]:
...

<details><summary><b>Ver uma resposta</b> (tente antes!)</summary>

```python
params_lento, _ = treinar(eta=0.05, passos=60)
params_lento   # ainda longe de 1.794: com passos pequenos, 60 iterações não bastam
```
</details>

## 6. scikit-learn: aprendizado de máquina em poucas linhas

Todo modelo do scikit-learn segue o mesmo roteiro (Aula 1):

```python
modelo = Modelo(opções)
modelo.fit(X_treino, y_treino)     # aprende
modelo.predict(X_novo)             # prevê
```

### 6.1 Prever a renda: regressão linear
Tarefa de **previsão**: estimar a renda de 1978 de um homem a partir do seu perfil. Usamos os 15.992 homens da CPS, que são muitos. Separamos 30% para **teste**: o modelo não vê esses dados no treino, e é neles que medimos o erro.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

X = cps[COVARIAVEIS] / [1, 1, 1, 1, 1, 1, 1000, 1000]   # rendas em mil US$
y = cps["re78"] / 1000

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)

# Referência ingênua: prever a média de todos
rmse_media = root_mean_squared_error(y_te, np.full(len(y_te), y_tr.mean()))

linear = LinearRegression().fit(X_tr, y_tr)
rmse_linear = root_mean_squared_error(y_te, linear.predict(X_te))

print(f"Erro típico (RMSE) prevendo a média: {rmse_media:.2f} mil US$")
print(f"Erro típico (RMSE) da regressão:     {rmse_linear:.2f} mil US$")
pd.Series(linear.coef_, index=COVARIAVEIS).round(3)

### 6.2 Trocar o modelo em uma linha: floresta aleatória
Uma **floresta aleatória** combina centenas de árvores de decisão e captura relações não lineares sem que precisemos especificá-las (Aula 5). O roteiro `fit`/`predict` é idêntico.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

floresta = RandomForestRegressor(n_estimators=300, min_samples_leaf=20,
                                 random_state=0, n_jobs=-1)
floresta.fit(X_tr, y_tr)
rmse_floresta = root_mean_squared_error(y_te, floresta.predict(X_te))

resultados = pd.Series({"Média": rmse_media, "Regressão linear": rmse_linear,
                        "Floresta aleatória": rmse_floresta})
resultados.round(3)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 3.8))

barras = axs[0].barh(resultados.index, resultados.values,
                     color=[CORES["cinza"], CORES["azul"], CORES["verde"]])
axs[0].bar_label(barras, fmt="%.2f", padding=3)
axs[0].set(title="Erro no teste (RMSE)", xlabel="mil US$")
axs[0].invert_yaxis()
axs[0].grid(axis="y", visible=False)

importancia = pd.Series(floresta.feature_importances_, index=COVARIAVEIS).sort_values()
importancia.plot.barh(ax=axs[1], color=CORES["laranja"])
axs[1].set(title="O que a floresta mais usa", xlabel="importância")
axs[1].grid(axis="y", visible=False)

fig.tight_layout()

**Lições honestas:**
- Os dois modelos cortam o erro da referência ingênua em mais de um quarto, mas a floresta **empata** com a regressão linear. Nem sempre o modelo mais sofisticado vence: compare sempre com alternativas simples.
- A renda passada (`re75`) é, de longe, o melhor preditor da renda futura.
- Um coeficiente ou uma importância **não** é efeito causal: são padrões úteis para prever.

O gráfico abaixo mostra por que nenhum modelo vai muito além: muita gente tem renda **zero** e muitas rendas da CPS estão **travadas num teto** (a pesquisa não divulga valores acima dele). Essas pilhas aparecem como as colunas verticais nas pontas.

In [ ]:
fig, ax = plt.subplots()
ax.scatter(y_te, floresta.predict(X_te), s=4, alpha=0.25, color=CORES["azul"])
ax.plot([0, 26], [0, 26], color=CORES["vermelho"], lw=1, label="previsão perfeita")
ax.set(title="Floresta: previsto × real (teste)", xlabel="renda real em 1978 (mil US$)",
       ylabel="renda prevista (mil US$)")
ax.legend()
print(f"{(cps['re78'] == cps['re78'].max()).mean():.0%} das rendas da CPS estão no teto "
      f"de US$ {cps['re78'].max():,.0f}")

### 6.3 Classificação: dá para prever quem foi tratado?
Uma pergunta de avaliação que o ML responde muito bem: **os grupos são comparáveis?** Se, olhando só para as covariáveis, um modelo consegue adivinhar quem foi tratado, os grupos são diferentes.

A **regressão logística** (o logit que vocês conhecem) prevê a probabilidade de ser tratado, a mesma ideia do **escore de propensão**. Vamos aplicá-la duas vezes: no experimento e na comparação com a CPS.

In [ ]:
from sklearn.linear_model import LogisticRegression


def prob_tratado(base):
    """Treina um logit de treat nas covariáveis e devolve, no teste,
    as probabilidades previstas e quem de fato foi tratado."""
    X = base[COVARIAVEIS] / [1, 1, 1, 1, 1, 1, 1000, 1000]
    X_tr, X_te, t_tr, t_te = train_test_split(X, base["treat"], test_size=0.3,
                                              random_state=0, stratify=base["treat"])
    logit = LogisticRegression(max_iter=5000).fit(X_tr, t_tr)
    return logit.predict_proba(X_te)[:, 1], t_te.to_numpy()


fig, axs = plt.subplots(1, 2, figsize=(10, 3.8), sharex=True)
for ax, (titulo, base) in zip(axs, [("Experimento (NSW)", nsw), ("Tratados × CPS", nao_exp)]):
    p, t = prob_tratado(base)
    bins = np.linspace(0, 1, 21)
    ax.hist(p[t == 0], bins=bins, density=True, alpha=0.6, color=CORES["cinza"], label="não tratados")
    ax.hist(p[t == 1], bins=bins, density=True, alpha=0.6, color=CORES["laranja"], label="tratados")
    ax.set(title=titulo, xlabel="probabilidade prevista de ser tratado")
    print(f"{titulo:18}: prob. média = {p[t == 1].mean():.2f} (tratados) × {p[t == 0].mean():.2f} (não tratados)")
axs[0].set_ylabel("densidade")
axs[0].legend()
fig.tight_layout()

- **No experimento**, o modelo **não consegue** separar os grupos: as distribuições se sobrepõem. É o sorteio funcionando.
- **Com a CPS**, quase todos os não tratados têm probabilidade perto de zero: o modelo separa os grupos com facilidade, porque eles são muito diferentes. É o diagnóstico do viés da Parte 4.

O aprendizado de máquina não substitui o desenho de identificação, mas é uma ótima ferramenta para **diagnosticar** e, como veremos na Aula 6, para **corrigir** comparações.

#### ✍️ Sua vez (extra)
Troque `LogisticRegression(max_iter=5000)` por `RandomForestClassifier(min_samples_leaf=10, random_state=0)` (importe de `sklearn.ensemble`) dentro de `prob_tratado` e rode de novo. A conclusão muda?

## 7. O que aprendemos

| Ferramenta | Recurso | Para quê |
|---|---|---|
| Python | tipos, listas, dicionários, `if`, `for`, `def`, *comprehensions* | lógica e organização do código |
| NumPy | arrays, vetorização, máscaras, `@`, `linalg.solve` | contas rápidas, álgebra matricial, MQO |
| pandas | `read_stata`, filtros, colunas novas, `groupby`, `concat` | ler, limpar e resumir bases |
| matplotlib | `subplots`, `hist`, `bar`, `bar_label`, `rcParams` | comunicar resultados |
| JAX | `grad` | derivadas exatas e gradiente descendente |
| scikit-learn | `fit`/`predict`, `train_test_split`, regressão, floresta, logit | prever e diagnosticar |

**Sobre a política:** com sorteio, a diferença de médias estima o efeito do programa (≈ US\$ 1.794). Sem sorteio, o grupo de comparação errado pode inverter o sinal.

**Para estudar em casa:** os cursos gratuitos do [Kaggle Learn](https://www.kaggle.com/learn) (*Python*, *Pandas*, *Intro to Machine Learning*) e os problemas do [LeetCode](https://leetcode.com/) para treinar lógica de programação.

**Referências:** LaLonde, R. (1986). Evaluating the Econometric Evaluations of Training Programs with Experimental Data. *American Economic Review*, 76(4), 604–620. Dehejia, R. e Wahba, S. (1999). Causal Effects in Nonexperimental Studies: Reevaluating the Evaluation of Training Programs. *Journal of the American Statistical Association*, 94(448), 1053–1062.